# Step 1 - Install required library

* Standard library for model calls
* Simplifies authentication and requests
* Avoids manual REST API handling
* Works smoothly in Google Colab

In [1]:
!pip -q install openai

# Step 2 - Import libraries

* userdata.get() fetches secrets securely
* json helps store/read structured state
* os.environ supports standard key handling
* Keeps the notebook clean and reusable

In [7]:
from openai import OpenAI
from google.colab import userdata
import os

# Step 3 - Load API key from Colab Secrets

* Prevents hardcoding secrets in notebooks
* Uses a standard environment-variable pattern
* Fails fast if the secret is missing
* Creates a reusable client for agent steps

In [14]:
openai_api_key = userdata.get("Eduhubspot_Key23")

if not openai_api_key:
    raise ValueError("Missing secret: Add 'Eduhubspot_Key23' in Colab Secrets and enable Notebook access.")

os.environ["OPENAI_API_KEY"] = openai_api_key
print("OpenAI key loaded from Colab Secrets")

OpenAI key loaded from Colab Secrets


# Step 4 - Initialize OpenAI Client

* The client automatically reads OPENAI_API_KEY
* One client instance is reused across the notebook
* This mirrors how Groq clients are typically initialized

In [12]:
client = OpenAI()
print("OpenAI client initialized successfully")

OpenAI client initialized successfully


# Step 5 - Minimal Test Call

* Confirms API key is valid
* Confirms network access
* Confirms model invocation works
* Prevents debugging later in agent logic

In [13]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input="Explain Agentic AI in one sentence."
)

print(response.output_text)

Agentic AI refers to artificial intelligence systems designed to autonomously make decisions and take actions to achieve specific goals without requiring constant human guidance.


# Step 6 - Create a reusable LLM helper function

* Makes code modular and easier to teach
* Encourages reuse across steps (plan/draft/check)
* Keeps prompts readable
* Returns clean text output for agent decisions

In [15]:
def call_llm(system_msg: str, user_msg: str, model: str = "gpt-4.1-mini") -> str:
    resp = client.responses.create(
        model=model,
        input=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
    )
    return resp.output_text.strip()

# Step 7 - Implement the Goal-Oriented Agent

* Demonstrates proactive behavior (goal pursuit)
* Implements a minimal agent state (goal/plan/draft/status)
* Uses a self-check gate (DONE/NOT_DONE) for control
* Limits iterations to manage cost and runtime

In [16]:
def goal_oriented_agent(goal: str, max_iters: int = 3) -> dict:
    state = {
        "goal": goal,
        "plan": None,
        "draft": None,
        "status": "NOT_DONE",
        "iterations_used": 0
    }

    # 1) PLAN
    state["plan"] = call_llm(
        system_msg="You are a planning assistant. Create short practical plans.",
        user_msg=(
            f"GOAL:\n{goal}\n\n"
            "Create a plan with 3–5 bullet steps. Keep it simple and actionable."
        )
    )

    # 2) LOOP: DRAFT -> CHECK -> REFINE
    for i in range(1, max_iters + 1):
        state["iterations_used"] = i

        # ACT (Draft)
        state["draft"] = call_llm(
            system_msg="You generate structured outputs that follow the goal strictly.",
            user_msg=(
                f"GOAL:\n{state['goal']}\n\n"
                f"PLAN:\n{state['plan']}\n\n"
                "Produce the final answer that satisfies the goal. Keep it structured."
            )
        )

        # EVALUATE (Goal check)
        verdict = call_llm(
            system_msg="You are a strict evaluator. Reply ONLY with DONE or NOT_DONE.",
            user_msg=(
                f"Check if the DRAFT fully satisfies the GOAL.\n\n"
                f"GOAL:\n{state['goal']}\n\n"
                f"DRAFT:\n{state['draft']}\n\n"
                "Reply with exactly one word: DONE or NOT_DONE."
            )
        )

        if verdict == "DONE":
            state["status"] = "DONE"
            break

        # REFINE plan if not done
        state["plan"] = call_llm(
            system_msg="You improve plans to better meet goals.",
            user_msg=(
                f"The draft did NOT fully meet the goal.\n\n"
                f"GOAL:\n{state['goal']}\n\n"
                f"OLD PLAN:\n{state['plan']}\n\n"
                "Rewrite an improved 3–5 bullet plan focusing on what was missing."
            )
        )

    return state

# Step 8 - Run the agent

* Shows the difference between plan and final delivery
* Demonstrates how self-check affects iterations
* Produces structured, student-friendly content
* Makes agent behavior visible (status + iteration count)

In [17]:
result = goal_oriented_agent(
    goal="Create a 3-day study plan to learn Agentic AI basics. Each day: 30 minutes. Include 1 mini task per day.",
    max_iters=3
)

print("STATUS:", result["status"])
print("\nPLAN:\n", result["plan"])
print("\nFINAL OUTPUT:\n", result["draft"])
print("\nITERATIONS USED:", result["iterations_used"])

STATUS: DONE

PLAN:
 **3-Day Study Plan: Agentic AI Basics (30 minutes/day)**

**Day 1: Introduction to Agentic AI**  
- Read a short article or watch a 10-minute video explaining what Agentic AI is.  
- Note down 3 key characteristics of Agentic AI.  
- Mini task: Write one example of an agentic AI application in real life.

**Day 2: Understanding How Agentic AI Works**  
- Study the basic components and decision-making process of Agentic AI.  
- Review a simple diagram or flowchart of an agent’s action cycle.  
- Mini task: Sketch your own flowchart of how an agent might make a decision.

**Day 3: Exploring Use Cases and Ethics**  
- Read about common use cases and potential ethical considerations of Agentic AI.  
- Summarize one benefit and one risk in your own words.  
- Mini task: Think of one question you have about Agentic AI for further study.

FINAL OUTPUT:
 **3-Day Study Plan: Agentic AI Basics (30 minutes/day)**

---

### Day 1: Introduction to Agentic AI  
- **Study Activit

# Step 9 (Optional) - Try a stricter goal to trigger refinement

* Demonstrates constraint handling under pressure
* Often forces at least one refinement iteration
* Reinforces goal satisfaction logic
* Mimics real enterprise requirement constraints

In [18]:
hard_goal = "Write a 90-word summary of Agentic AI with exactly 3 bullet points and 1 limitation."
result2 = goal_oriented_agent(hard_goal, max_iters=3)

print("STATUS:", result2["status"])
print("\nFINAL OUTPUT:\n", result2["draft"])
print("\nITERATIONS USED:", result2["iterations_used"])

STATUS: DONE

FINAL OUTPUT:
 **Agentic AI Summary**

Agentic AI refers to artificial intelligence systems capable of autonomous decision-making and goal-directed behavior, resembling human-like agency.

- It can perceive environments, set objectives, and plan actions independently.  
- Agentic AI enhances adaptability by learning from experiences and adjusting strategies.  
- Applications include robotics, autonomous vehicles, and intelligent assistants improving efficiency and user interaction.

**Limitation:**  
A key challenge is ensuring ethical behavior and preventing unintended consequences due to unpredictable autonomous decisions.

(Word count: 90)

ITERATIONS USED: 1
